---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Topic**: You Can Just Build Things

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

## Welcome!

In our firstfour lectures, we've covered how
1. We can call LLMs via APIs and get structured responses
2. We can build lexical search with BM25
3. We can build semantic search with embeddings
4. We can combine lexical and semantic search into hybrid search

Today you will put it all together by building a Retrieval Augmented Generation (RAG) system.
- This is a question-answering bot that can answer questions about Fordham University
- You will use real data scraped from the Fordham website.


Your RAG pipeline will look like this:

```
User Question
     ↓
1. RETRIEVE: Find relevant documents (search!)
     ↓
2. AUGMENT: Stuff those documents into a prompt
     ↓
3. GENERATE: Ask an LLM to answer using the context
     ↓
Answer
```


---

# 1. Look at your data

In `data/fordham-website.zip` you'll find **~9,500 Markdown files** scraped from Fordham's website. Each file is one page — admissions info, program descriptions, faculty pages, financial aid, campus life, and more.

Your task: **look at the data**
- The first step in any AI engineering or data science project should always be to familiarize yourself with the data.
- I cannot stress this enough.. without this step, it's hard to build anything useful.

Tips:
- Unzip the archive and look at some of the files. 
- Open a few in a text editor. 
- Get a feel for what you're working with.
- The first line of every file is always the **URL** of the page it was scraped from. The rest is the page content converted to Markdown. Here's an example — `gabelli-school-of-business_veterans.md`:

```markdown
https://www.fordham.edu/gabelli-school-of-business/veterans

# Military Veterans & Active Duty Members of the Military

## Transform Your Knowledge & Skills Into a Business Career for the Future

As a veteran or an active duty member of the United States Armed Services,
you have gained or are currently acquiring the invaluable organizational,
leadership, analytics, and technical knowledge and skills that hiring
managers seek. These transferrable skills provide a major advantage in
emerging, business-related industries where innovation, a global mind-set,
and the ability to lead individuals and teams in the continuously evolving
work environment, are critical for success.

By completing a graduate or undergraduate business degree at the Gabelli
School of Business, you can prepare for a lifelong career in some of
today's fastest-growing fields. ...

### Study at a Top-Ranked, Military-Friendly University

The Gabelli School of Business is part of Fordham University, the only
New York City university to be among those ranked "Best for Vets" by
Military Times. ...

### Learn How the Yellow Ribbon Program Works

The Yellow Ribbon GI Education Enhancement Program, or the Yellow Ribbon
Program, is a part of the Post-9/11 Veterans Educational Assistance Act
of 2008. ...
```

The filenames mirror the URL structure — underscores replace path separators (e.g. `gabelli-school-of-business_veterans.md` came from `/gabelli-school-of-business/veterans`). Some files are short (a few lines), others are quite long.

- Once you've looked around, load the files into Python. Python's built-in `zipfile` module can read zip archives without extracting to disk. Load them into a list of dictionaries or a DataFrame with at least two fields: the filename (or a clean page name) and the content

In [1]:
import pandas as pd
pd.__version__


'2.3.3'

In [4]:
from pathlib import Path

data_files = sorted([p.name for p in Path("data").iterdir()])
data_files


FileNotFoundError: [Errno 2] No such file or directory: 'data'

In [3]:
import zipfile
import pandas as pd
import os
import pathlib

def load_fordham_data(source_path):
    data = []
    
    # 1. Check if it's a zip file
    if source_path.endswith('.zip') and os.path.exists(source_path):
        print(f"Loading from zip: {source_path}")
        with zipfile.ZipFile(source_path, 'r') as z:
            file_list = [f for f in z.namelist() if f.endswith('.md')]
            for file_name in file_list:
                with z.open(file_name) as f:
                    try:
                        content = f.read().decode('utf-8')
                    except UnicodeDecodeError:
                        continue
                    
                    lines = content.split('\n', 1)
                    url = lines[0].strip() if lines else ""
                    body = lines[1].strip() if len(lines) > 1 else ""
                    
                    data.append({
                        "filename": file_name,
                        "url": url,
                        "content": body
                    })
                    
    # 2. Check if it's a directory (Robust Fallback)
    elif os.path.exists(source_path) and os.path.isdir(source_path):
        print(f"Loading from directory: {source_path}")
        path = pathlib.Path(source_path)
        files = list(path.glob('*.md'))
        print(f"Found {len(files)} markdown files.")
        
        for file_path in files:
             try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                    if not lines: continue
                    
                    url = lines[0].strip()
                    content = "".join(lines[1:]).strip()
                    
                    data.append({
                        "filename": file_path.name,
                        "url": url,
                        "content": content
                    })
             except Exception as e:
                print(f"Error reading {file_path}: {e}")
                
    else:
        print(f"Source not found or invalid: {source_path}")
        # Try default locations if provided path fails
        fallback_dir = 'data/fordham-website'
        if source_path != fallback_dir and os.path.exists(fallback_dir) and os.path.isdir(fallback_dir):
             print(f"Fallback: Loading from {fallback_dir}")
             return load_fordham_data(fallback_dir) # Recursive call
             
        return pd.DataFrame()

    return pd.DataFrame(data)

# Usage - Point to the directory since you extracted it
source_path = 'data/fordham-website' 
df = load_fordham_data(source_path)

print(f"Loaded {len(df)} documents.")
print(df.head())

Source not found or invalid: data/fordham-website
Loaded 0 documents.
Empty DataFrame
Columns: []
Index: []


---

# 2. Chunk the Documents

Some of the pages could be too long to embed as a single unit. Down the line, the pages may be too long to stuff into the LLM's prompt during the generation step. As such, most of the RAG systems will break down big documents into into smaller **chunks**.

> 📚 **TERM: Chunking**  
> Splitting documents into smaller, self-contained pieces for embedding and retrieval. The goal is chunks that are small enough to be specific, but large enough to be meaningful.

Your task: **write a function that splits each document into chunks.**

Things to think about:
- What's a reasonable chunk size? (Think about what fits in a prompt vs. what's too vague)
- Should you split on sentences? Paragraphs? A fixed character/word count?
- Should chunks overlap? What happens if an answer spans two chunks?
- How do you keep track of which document each chunk came from? You may need that information down the line.

In [ ]:
# Placeholder for your implementation
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    if not text:
        return chunks
        
    start = 0
    text_len = len(text)
    
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        # Move the start pointer forward
        step = chunk_size - overlap
        if step <= 0: step = 1 # Avoid infinite loop
        start += step
        
    return chunks

def process_to_chunks(df):
    chunked_data = []
    
    for index, row in df.iterrows():
        # Get content, default to empty string if missing
        content = row.get('content', '')
        if not isinstance(content, str): content = ""
            
        # Split the body content into pieces
        text_chunks = chunk_text(content)
        
        for i, chunk_content in enumerate(text_chunks):
            # Only add non-empty chunks if you prefer
            if not chunk_content.strip(): continue
                
            chunked_data.append({
                "chunk_id": f"{row['filename']}_{i}",
                "source_url": row['url'],
                "content": chunk_content.strip(),
                "parent_file": row['filename']
            })
            
    return pd.DataFrame(chunked_data)

# Apply the function to create df_chunks
df_chunks = process_to_chunks(df)

print(f"Generated {len(df_chunks)} chunks from {len(df)} documents.")
print(df_chunks.head())

Generated 66082 chunks from 9530 documents.
                                            chunk_id  \
0  about_living-the-mission_campus-ministry_catho...   
1  about_living-the-mission_campus-ministry_catho...   
2  about_living-the-mission_campus-ministry_catho...   
3  about_living-the-mission_campus-ministry_catho...   
4  about_living-the-mission_campus-ministry_catho...   

                                          source_url  \
0  https://www.fordham.edu/about/living-the-missi...   
1  https://www.fordham.edu/about/living-the-missi...   
2  https://www.fordham.edu/about/living-the-missi...   
3  https://www.fordham.edu/about/living-the-missi...   
4  https://www.fordham.edu/about/living-the-missi...   

                                             content  \
0  # Ministry of Music\n\n\nFordham offers each s...   
1  itan area over the live radio broadcast by WFU...   
2  mble:**Liturgical/Worship - Mixed Voices**Memb...   
3  ers at the 7:00 p.m. Sunday Mass in the Univer...   
4 

---

# 3. Embed the Chunks

Now we need to turn each chunk into a vector so we can search over them. You've done this before in Lecture 4.

Your task: **embed all chunks using an embedding model.**

Tips:
- You could use a local model, or API model. What are the tradeoffs?
- This will take a while if you do it serially. You might want to use async/batch.
- Once you've created your embeddings, you may want to save them to disk so you don't have to redo this step every time
- You'll need to embed queries with the **same model** at search time

In [ ]:
# Placeholder for your implementation
from sentence_transformers import SentenceTransformer
import numpy as np

# 1. Initialize the model
# 'all-MiniLM-L6-v2' is a small, fast model great for local use
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Get the list of texts to embed
# We use the 'content' column of our chunks dataframe
chunk_texts = df_chunks['content'].tolist()

print(f"Embedding {len(chunk_texts)} chunks...")

# 3. Embed the chunks
# The model handles batching automatically
embeddings = model.encode(chunk_texts, show_progress_bar=True)

# 4. Add embeddings to the dataframe 
df_chunks['embedding'] = list(embeddings)

print("Embedding complete.")
print(df_chunks.head())

Embedding 66082 chunks...


Batches: 100%|██████████| 2066/2066 [05:24<00:00,  6.37it/s]


Embedding complete.
                                            chunk_id  \
0  about_living-the-mission_campus-ministry_catho...   
1  about_living-the-mission_campus-ministry_catho...   
2  about_living-the-mission_campus-ministry_catho...   
3  about_living-the-mission_campus-ministry_catho...   
4  about_living-the-mission_campus-ministry_catho...   

                                          source_url  \
0  https://www.fordham.edu/about/living-the-missi...   
1  https://www.fordham.edu/about/living-the-missi...   
2  https://www.fordham.edu/about/living-the-missi...   
3  https://www.fordham.edu/about/living-the-missi...   
4  https://www.fordham.edu/about/living-the-missi...   

                                             content  \
0  # Ministry of Music\n\n\nFordham offers each s...   
1  itan area over the live radio broadcast by WFU...   
2  mble:**Liturgical/Worship - Mixed Voices**Memb...   
3  ers at the 7:00 p.m. Sunday Mass in the Univer...   
4  #### What type of music

---

# 4. Retrieve

Now build the **R** in RAG. Given a user's question, find the most relevant chunks.

Your task: **write a retrieval function that takes a question and returns the most relevant chunks.**

Tips:
- You can use lexical or semantic search or both!
- How many chunks should you retrieve? Too few and you might miss the answer; too many and you'll overwhelm the LLM (and pay more tokens)
- Try a few test questions and eyeball whether the retrieved chunks are relevant
- Try a few questions and see what comes back. For example:
  - "What programs does the Gabelli School of Business offer?"
  - "How do I apply for financial aid?"
  - "Where is Fordham's campus?"

In [ ]:
import numpy as np

# Precompute once
emb_matrix = np.stack(df_chunks["embedding"].values).astype("float32")
emb_matrix = emb_matrix / np.linalg.norm(emb_matrix, axis=1, keepdims=True)

def retrieve(query, top_k=5):
    q = model.encode([query])[0].astype("float32")
    q = q / np.linalg.norm(q)

    scores = emb_matrix @ q
    top_idx = np.argsort(scores)[::-1][:top_k]
    out = df_chunks.iloc[top_idx].copy()
    out["score"] = scores[top_idx]
    return out


In [ ]:
test_queries = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid?",
    "Where is Fordham's campus?"
]
for q in test_queries:
    print(f"\nQuestion: {q}")
    results = retrieve(q, top_k=5)
    for _, row in results.iterrows():
        print(f"  - ({row['score']:.3f}) [{row['parent_file']}] {row['source_url']} | {row['content'][:120]}...")


NameError: name 'test_queries' is not defined

---

# 5. Generate

Now build the **G** in RAG. Take the retrieved chunks and pass them to an LLM along with the user's question.

Your task: **write a function that takes a question and the retrieved chunks, builds a prompt, and calls an LLM to generate an answer.**

Tips:
- How should you structure the prompt? The LLM needs to know: (1) what is the context of the application, (2) what is the question, (3) what it should include in its answer
- What should the LLM do if the context doesn't contain the answer?
- Start with a cheap model; try a better one when you've figured out the pipeline

In [ ]:
# Placeholder for your implementation

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI()


In [ ]:
def build_prompt(question, retrieved_df):
    context_blocks = []

    for _, row in retrieved_df.iterrows():
        block = f"""
SOURCE: {row['source_url']}
{row['content']}
"""
        context_blocks.append(block)

    context = "\n---\n".join(context_blocks)

    prompt = f"""
You are a Fordham University information assistant.

Use ONLY the context below to answer the question.
If the answer is not in the context, say:
"I don't know based on the provided context."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""
    return prompt


In [ ]:
def generate_answer(question, retrieved_df, model="gpt-4o-mini"):
    prompt = build_prompt(question, retrieved_df)

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400
    )

    return response.choices[0].message.content



In [ ]:
def rag(question, top_k=5):
    retrieved = retrieve(question, top_k=top_k)
    answer = generate_answer(question, retrieved)

    return answer


In [ ]:
print(rag("How do I apply for financial aid?"))


To apply for financial aid at Fordham University, applicants and their families should fill out both the Free Application for Federal Student Aid (FAFSA) to be considered for federal aid, and the CSS Profile to be considered for institutional aid. Additionally, it's important to check the specific application deadlines based on your choice of admission application (Early Action, Early Decision, or Regular Decision).


---

# 6. Wire everything together

Combine the previous steps into a simple function that takes in a question and returns an answer.

Your task: **write a `rag(question)` function that retrieves relevant chunks and generates an answer.**

In [ ]:
# Placeholder for your implementation
def rag(question, top_k=5):
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks
    2. Generate answer using LLM
    3. Return answer
    """
    
    # Step 1: retrieve
    retrieved_chunks = retrieve(question, top_k=top_k)
    
    # Step 2: generate
    answer = generate_answer(question, retrieved_chunks)
    
    # Step 3: return
    return answer


In [ ]:
questions = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid?",
    "Where are Fordham campuses located?"
]

for q in questions:
    print("\n" + "="*70)
    print("Q:", q)
    print("\nA:\n", rag(q))



Q: What programs does the Gabelli School of Business offer?

A:
 The Gabelli School of Business offers a range of graduate and executive programs, including three types of M.B.A. programs (full-time, professional, and others). All students complete a liberal arts core, a business core, a major and concentration, and electives.

Q: How do I apply for financial aid?

A:
 To apply for financial aid at Fordham University, you should fill out both the Free Application for Federal Student Aid (FAFSA) to be considered for federal aid and the CSS Profile to be considered for institutional aid. Make sure to check the specific deadlines based on your choice of admission application (Early Action, Early Decision, or Regular Decision).

Q: Where are Fordham campuses located?

A:
 Fordham University has three campuses located at:

1. **Rose Hill campus** - Northern Bronx, adjacent to the New York Botanical Garden.
2. **Lincoln Center campus** - Manhattan, located at 113 W. 60th Street (at Columbus

In [ ]:
def rag(question, top_k=5, return_sources=False):
    retrieved = retrieve(question, top_k)
    answer = generate_answer(question, retrieved)

    if return_sources:
        return answer, retrieved["source_url"].tolist()

    return answer


---

# 7. Evaluate, experiment and improve

Your RAG system works — but there's always room to make it better. 

Your task: **evaluate, experiment, and improve your system**

Tips:
- How do you know that your system is working or that your changes are improving it?
- Try different questions — where does it do well? Where does it struggle?
- Adjust the number of retrieved chunks — what happens with more or fewer?
- Try different chunking strategies — bigger chunks? Smaller? Overlap?
- Try a different embedding model — does it change retrieval quality?
- Improve the prompt — can you get better, more concise answers?
- Add source attribution — can the system tell the user which pages the answer came from?

In [ ]:
# Placeholder for your implementation
eval_questions = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid?",
    "Where are Fordham campuses located?",
    "What is undergraduate tuition?",
    "How can I contact admissions?",
    "What housing options are available?"
]



In [ ]:
def evaluate_system(top_k=5):
    print(f"\n===== Testing with top_k = {top_k} =====\n")

    for q in eval_questions:
        print("="*70)
        print("Q:", q)
        ans = rag(q, top_k=top_k)
        print("A:", ans[:500], "\n")


In [ ]:
evaluate_system(top_k=3)
evaluate_system(top_k=5)
evaluate_system(top_k=8)



===== Testing with top_k = 3 =====

Q: What programs does the Gabelli School of Business offer?
A: The Gabelli School of Business offers a range of graduate and executive programs, including three types of M.B.A. programs (full-time, professional, and executive). Students also complete a liberal arts core, a business core, a major and concentration, and electives as part of their curriculum. 

Q: How do I apply for financial aid?
A: To apply for financial aid, you should fill out both the Free Application for Federal Student Aid (FAFSA) to be considered for federal aid and the CSS Profile to be considered for institutional aid. Additionally, ensure you have been accepted and enrolled as a matriculated student in an approved program of study. Be sure to check the financial aid application deadlines based on your choice of admission application (Early Action, Early Decision, or Regular Decision). 

Q: Where are Fordham campuses located?
A: Fordham University has campuses located in:

1.

In [ ]:
chunk_size=1200; overlap=200


SyntaxError: cannot assign to literal (6889175.py, line 1)

In [ ]:
def build_prompt(question, retrieved_df):
    context = "\n\n---\n\n".join(retrieved_df["content"].tolist())

    return f"""
You are a Fordham University assistant.

Rules:
- Answer ONLY using the context
- Be concise (3-5 sentences max)
- If unsure, say you don't know
- Cite sources at the end

Context:
{context}

Question: {question}

Answer:
"""


---

# 8. (Optional) Make it an app

So far your RAG system lives inside a notebook. That's great for development — but nobody is going to use your Jupyter notebook to ask questions about Fordham. Let's turn it into a real web app.

> 📚 **TERM: Streamlit**  
> A Python library that turns plain Python scripts into interactive web apps. You write Python — no HTML, CSS, or JavaScript — and Streamlit renders it as a web page with inputs, buttons, and formatted output. It's the fastest way to go from "I have a function" to "I have a web app."

Your task: **create a Streamlit app that lets a user type a question about Fordham and get an answer from your RAG system.**

To get started:
- Install it: `uv pip install streamlit` 
- A Streamlit app is just a `.py` file (not a notebook). Create something like `fordham_rag_app.py`
- Run it: `streamlit run scripts/fordham_rag_app.py` — this opens a browser tab with your app

Tips:
- Check out the [Streamlit docs](https://docs.streamlit.io/) — the "Get started" tutorial is very short
- Your best bet is to vibecode your way to this. You'll be surprised how fast you can get it up and running

In [ ]:
# Placeholder for your implementation

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI()


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

st.set_page_config(page_title="Fordham RAG", layout="wide")
st.title("Fordham RAG Assistant")

# Paths (project root is parent of scripts/)
PROJECT_ROOT = Path.cwd().resolve().parents[0]
CHUNKS_PATH = PROJECT_ROOT / "chunks.csv"
EMB_PATH = PROJECT_ROOT / "chunk_embeddings.npy"
ENV_PATH = PROJECT_ROOT / ".env"

# Show immediate UI so page is never blank
st.write("Project root:", str(PROJECT_ROOT))
st.write("Chunks file:", str(CHUNKS_PATH))
st.write("Embeddings file:", str(EMB_PATH))

# Load env
load_dotenv(dotenv_path=ENV_PATH)

if not os.getenv("OPENAI_API_KEY"):
    st.error("OPENAI_API_KEY missing. Add it to .env in the project root.")
    st.stop()

client = OpenAI()

def load_index():
    if not CHUNKS_PATH.exists():
        raise FileNotFoundError(f"chunks.csv not found at {CHUNKS_PATH}")
    if not EMB_PATH.exists():
        raise FileNotFoundError(f"chunk_embeddings.npy not found at {EMB_PATH}")

    # Load chunks (CSV can be large)
    df = pd.read_csv(CHUNKS_PATH)

    # Load embeddings efficiently
    emb = np.load(EMB_PATH, mmap_mode="r").astype(np.float32)

    # Normalize (if already normalized, this is still safe)
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)

    return df, emb

def embed_query(q: str) -> np.ndarray:
    resp = client.embeddings.create(model=EMBED_MODEL, input=[q])
    v = np.array(resp.data[0].embedding, dtype=np.float32)
    v = v / (np.linalg.norm(v) + 1e-12)
    return v

def retrieve(df: pd.DataFrame, emb: np.ndarray, question: str, top_k: int):
    qv = embed_query(question)
    scores = emb @ qv
    idx = np.argsort(-scores)[:top_k]
    out = df.iloc[idx].copy()
    out["score"] = scores[idx]
    return out

def generate_answer(question: str, sources_df: pd.DataFrame) -> str:
    blocks = []
    for i, row in sources_df.reset_index(drop=True).iterrows():
        url = str(row.get("url", "")).strip()
        chunk = str(row.get("chunk", "")).strip()
        blocks.append(f"Source {i+1}\nURL: {url}\n\n{chunk}")

    context = "\n\n---\n\n".join(blocks)

    prompt = f"""Answer using ONLY the sources. If not found, say: I don't know based on the provided sources.
Cite sources like [Source 1], [Source 2].

Question: {question}

Sources:
{context}
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return resp.choices[0].message.content

with st.sidebar:
    top_k = st.slider("Top K sources", 3, 12, 5)
    show_sources = st.checkbox("Show sources", True)

question = st.text_input("Ask a question about Fordham", placeholder="e.g., How do I apply for financial aid?")
ask = st.button("Ask")

if ask:
    if not question.strip():
        st.warning("Please type a question.")
        st.stop()

    with st.spinner("Loading index (chunks + embeddings)..."):
        try:
            df_chunks, vectors = load_index()
        except Exception as e:
            st.error("Failed to load data files.")
            st.exception(e)
            st.stop()

    st.success(f"Loaded {len(df_chunks)} chunks. Embeddings shape: {vectors.shape}")

    with st.spinner("Retrieving sources..."):
        sources = retrieve(df_chunks, vectors, question, top_k=top_k)

    with st.spinner("Generating answer..."):
        ans = generate_answer(question, sources)

    st.subheader("Answer")
    st.write(ans)

    if show_sources:
        st.subheader("Sources")
        for i, row in sources.reset_index(drop=True).iterrows():
            st.markdown(f"**Source {i+1}** | score `{row['score']:.4f}`")
            url = str(row.get("url", "")).strip()
            if url:
                st.write(url)
            st.code(str(row.get("chunk", ""))[:1200])
            st.divider()


2026-02-11 16:02:43.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-11 16:02:43.618 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-11 16:02:43.619 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-11 16:02:43.619 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NameError: name '__file__' is not defined

---

# Summary

## What You Built

| Step | What You Did | What It Does |
|------|-------------|-------------|
| **Load** | Read 9,500+ Fordham web pages | Get raw content |
| **Chunk** | Split pages into smaller pieces | Make content searchable and promptable |
| **Embed** | Turn chunks into vectors | Enable semantic search |
| **Retrieve** | Find relevant chunks for a question | The **R** in RAG |
| **Generate** | Ask an LLM to answer using the chunks | The **G** in RAG |
| **RAG** | Wire it all together | Question in, answer out |

## The Big Picture

RAG is one of the most common patterns in AI engineering today. What you built here is the same core architecture behind tools like ChatGPT with search, Perplexity, enterprise Q&A bots, and more. The details get more sophisticated (vector databases, reranking, query rewriting, evaluation) but the pattern is the same:

**Find relevant stuff → give it to an LLM → get an answer.**

You can just build things.